# EMCAD Experiment 2 — BraTS Preparation
## Part 1: Download, verify, split, audit, and build a compact 2D cache

This notebook performs **data preparation only**. It does **not** train EMCAD and does not need a GPU. The official `dataset.json` training list is used as the canonical source of the 484 labelled cases; any extra archive files are documented and ignored.

### Source
Medical Segmentation Decathlon — `Task01_BrainTumour`

Direct public mirror:
`https://msd-for-monai.s3-us-west-2.amazonaws.com/Task01_BrainTumour.tar`

Expected MD5:
`240a19d752f0d9e9101544901065d872`

The task contains the BraTS 2016/2017 brain-tumour cases. We use only the 484 labelled cases.

### Fixed experimental protocol
- Patient-level split: **388 train / 48 validation / 48 test**
- Split seed: **2026**
- Binary target: **Whole Tumour (WT) = label > 0**
- Input channels for 2D EMCAD: **FLAIR + T1-Gd + T2**
- No patient leakage
- Cache per train/validation patient:
  - up to 8 tumour-positive axial slices
  - up to 2 tumour-negative brain slices
- Test patients are packaged as original NIfTI volumes for later full-volume evaluation.

### Kaggle settings
- **Accelerator: None / GPU OFF**
- **Internet: ON**
- **No input dataset is required**




In [1]:
from pathlib import Path
import subprocess
import sys

print("Python:", sys.version.split()[0])

try:
    import nibabel
    import numpy
    import torch

    print("PyTorch:", torch.__version__)
    print("NumPy:", numpy.__version__)
    print("Nibabel:", nibabel.__version__)

except ImportError:
    print("Installing nibabel...")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "nibabel",
        ],
        check=True,
    )

    import nibabel
    import numpy
    import torch

    print("PyTorch:", torch.__version__)
    print("NumPy:", numpy.__version__)
    print("Nibabel:", nibabel.__version__)


Python: 3.13.15
PyTorch: 2.11.0+cpu
NumPy: 2.1.3
Nibabel: 5.4.2


In [2]:
from pathlib import Path

prep_code = '\nfrom pathlib import Path\nimport csv\nimport hashlib\nimport json\nimport os\nimport random\nimport shutil\nimport subprocess\nimport tarfile\nimport time\n\nimport nibabel as nib\nimport numpy as np\nimport torch\n\n\n# ---------------------------------------------------------------------\n# Configuration\n# ---------------------------------------------------------------------\n\nWORK = Path("/kaggle/working")\n\nDOWNLOAD_URL = (\n    "https://msd-for-monai.s3-us-west-2.amazonaws.com/"\n    "Task01_BrainTumour.tar"\n)\n\nEXPECTED_MD5 = "240a19d752f0d9e9101544901065d872"\n\nARCHIVE = WORK / "Task01_BrainTumour.tar"\nDATA_PARENT = WORK / "brats_msd"\nDATA_ROOT = DATA_PARENT / "Task01_BrainTumour"\n\nPREP_ROOT = WORK / "EMCAD_BraTS_Part1"\nCACHE_ROOT = PREP_ROOT / "slice_cache"\nTEST_ROOT = PREP_ROOT / "test_volumes"\n\nSPLIT_SEED = 2026\nTRAIN_CASES = 388\nVAL_CASES = 48\nTEST_CASES = 48\n\nPOSITIVE_SLICES_PER_PATIENT = 8\nNEGATIVE_SLICES_PER_PATIENT = 2\n\n\n# ---------------------------------------------------------------------\n# General utilities\n# ---------------------------------------------------------------------\n\ndef md5sum(path):\n    h = hashlib.md5()\n\n    with path.open("rb") as f:\n        for block in iter(lambda: f.read(4 * 1024 * 1024), b""):\n            h.update(block)\n\n    return h.hexdigest()\n\n\ndef sha256sum(path):\n    h = hashlib.sha256()\n\n    with path.open("rb") as f:\n        for block in iter(lambda: f.read(4 * 1024 * 1024), b""):\n            h.update(block)\n\n    return h.hexdigest()\n\n\ndef case_id(path):\n    name = Path(path).name\n\n    if name.endswith(".nii.gz"):\n        return name[:-7]\n\n    return Path(name).stem\n\n\ndef write_csv(path, rows):\n    if not rows:\n        return\n\n    with path.open("w", newline="") as f:\n        writer = csv.DictWriter(f, fieldnames=rows[0].keys())\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef evenly_spaced(indices, count):\n    indices = list(indices)\n\n    if len(indices) <= count:\n        return indices\n\n    positions = torch.linspace(\n        0,\n        len(indices) - 1,\n        steps=count,\n    ).round().long()\n\n    selected = []\n\n    for position in positions.tolist():\n        value = indices[position]\n\n        if value not in selected:\n            selected.append(value)\n\n    return selected\n\n\ndef safe_extract(tar, members, destination):\n    destination = destination.resolve()\n\n    for member in members:\n        normalized = member.name.lstrip("./")\n        member.name = normalized\n\n        target = (destination / normalized).resolve()\n\n        if not str(target).startswith(str(destination)):\n            raise RuntimeError(f"Unsafe archive path: {normalized}")\n\n        tar.extract(member, destination)\n\n\n# ---------------------------------------------------------------------\n# 1. Download and verify\n# ---------------------------------------------------------------------\n\ndef download_dataset():\n    DATA_PARENT.mkdir(parents=True, exist_ok=True)\n\n    extracted_ready = (\n        (DATA_ROOT / "dataset.json").is_file()\n        and (DATA_ROOT / "imagesTr").is_dir()\n        and (DATA_ROOT / "labelsTr").is_dir()\n    )\n\n    if extracted_ready:\n        print("Verified extracted labelled data already exists.")\n        print("Skipping the 7.09 GiB download in this session.")\n        return\n\n    if not ARCHIVE.exists():\n        free_gb = shutil.disk_usage(WORK).free / 1024 ** 3\n\n        print(f"Free disk before download: {free_gb:.2f} GiB")\n        print("Downloading:")\n        print(DOWNLOAD_URL)\n\n        subprocess.run(\n            [\n                "curl",\n                "-fL",\n                "--retry", "12",\n                "--retry-all-errors",\n                "--retry-delay", "5",\n                "--connect-timeout", "30",\n                "--continue-at", "-",\n                DOWNLOAD_URL,\n                "-o", str(ARCHIVE),\n            ],\n            check=True,\n        )\n\n    actual_md5 = md5sum(ARCHIVE)\n\n    print("Archive size:", f"{ARCHIVE.stat().st_size / 1024 ** 3:.2f} GiB")\n    print("Archive MD5:", actual_md5)\n\n    if actual_md5 != EXPECTED_MD5:\n        raise RuntimeError(\n            "Dataset checksum mismatch.\\n"\n            f"Expected: {EXPECTED_MD5}\\n"\n            f"Actual:   {actual_md5}"\n        )\n\n    print("Dataset archive verification: PASS")\n\n\n# ---------------------------------------------------------------------\n# 2. Extract only labelled data\n# ---------------------------------------------------------------------\n\ndef extract_labelled_data():\n    if DATA_ROOT.is_dir():\n        print("Labelled data already extracted:", DATA_ROOT)\n        return\n\n    print("Extracting imagesTr, labelsTr, and dataset.json...")\n\n    with tarfile.open(ARCHIVE, "r") as tar:\n        members = []\n\n        for member in tar.getmembers():\n            name = member.name.lstrip("./")\n\n            if (\n                name == "Task01_BrainTumour/dataset.json"\n                or name.startswith("Task01_BrainTumour/imagesTr/")\n                or name.startswith("Task01_BrainTumour/labelsTr/")\n            ):\n                members.append(member)\n\n        if not members:\n            raise RuntimeError(\n                "No labelled Task01_BrainTumour members found in archive."\n            )\n\n        safe_extract(tar, members, DATA_PARENT)\n\n    if not DATA_ROOT.is_dir():\n        raise RuntimeError("Extraction failed.")\n\n    print("Labelled-data extraction: PASS")\n\n    # Recover disk space immediately.\n    ARCHIVE.unlink()\n\n    print("Downloaded archive deleted after verified extraction.")\n\n\n# ---------------------------------------------------------------------\n# 3. Inspect dataset metadata\n# ---------------------------------------------------------------------\n\ndef read_metadata():\n    metadata_path = DATA_ROOT / "dataset.json"\n\n    if not metadata_path.is_file():\n        raise FileNotFoundError(metadata_path)\n\n    metadata = json.loads(metadata_path.read_text())\n\n    print("Dataset:", metadata.get("name"))\n    print("Declared training cases:", metadata.get("numTraining"))\n    print("Modalities:", metadata.get("modality"))\n    print("Labels:", metadata.get("labels"))\n\n    return metadata\n\n\ndef resolve_channels(metadata):\n    modality = metadata.get("modality", {})\n\n    if len(modality) != 4:\n        raise RuntimeError(\n            f"Expected 4 MRI modalities, found: {modality}"\n        )\n\n    normalized = {}\n\n    for key, value in modality.items():\n        text = str(value).lower()\n        text = (\n            text.replace("-", "")\n            .replace("_", "")\n            .replace(" ", "")\n        )\n        normalized[int(key)] = text\n\n    flair = None\n    t1gd = None\n    t2 = None\n\n    for index, name in normalized.items():\n        if "flair" in name:\n            flair = index\n        elif (\n            "t1gd" in name\n            or "t1ce" in name\n            or "t1c" in name\n            or ("t1" in name and "gd" in name)\n        ):\n            t1gd = index\n        elif name.startswith("t2"):\n            t2 = index\n\n    if flair is None or t1gd is None or t2 is None:\n        raise RuntimeError(\n            "Could not resolve FLAIR/T1-Gd/T2 from dataset metadata."\n        )\n\n    selected = {\n        "FLAIR": flair,\n        "T1-Gd": t1gd,\n        "T2": t2,\n    }\n\n    print("Selected EMCAD input channels:", selected)\n\n    return selected\n\n\n# ---------------------------------------------------------------------\n# 4. Collect all labelled patients\n# ---------------------------------------------------------------------\n\ndef collect_cases(metadata):\n    """\n    Use dataset.json as the canonical source of labelled training cases.\n\n    The downloaded archive can contain additional NIfTI files in imagesTr\n    that are not members of the official 484-case training list. Those files\n    are recorded for provenance but are never used in this experiment.\n    """\n    training = metadata.get("training")\n\n    if not isinstance(training, list):\n        raise RuntimeError(\n            "dataset.json does not contain a valid \'training\' list."\n        )\n\n    declared = int(metadata.get("numTraining", len(training)))\n\n    if declared != 484 or len(training) != 484:\n        raise RuntimeError(\n            "Expected dataset.json to define exactly 484 labelled cases, "\n            f"but numTraining={declared} and len(training)={len(training)}."\n        )\n\n    images = {}\n    labels = {}\n    canonical_image_paths = set()\n    canonical_label_paths = set()\n\n    for entry in training:\n        if not isinstance(entry, dict):\n            raise RuntimeError(\n                f"Invalid dataset.json training entry: {entry}"\n            )\n\n        image_rel = str(entry.get("image", "")).lstrip("./")\n        label_rel = str(entry.get("label", "")).lstrip("./")\n\n        image_path = DATA_ROOT / image_rel\n        label_path = DATA_ROOT / label_rel\n\n        if not image_path.is_file():\n            raise FileNotFoundError(\n                f"Canonical training image is missing: {image_path}"\n            )\n\n        if not label_path.is_file():\n            raise FileNotFoundError(\n                f"Canonical training label is missing: {label_path}"\n            )\n\n        image_id = case_id(image_path)\n        label_id = case_id(label_path)\n\n        if image_id != label_id:\n            raise RuntimeError(\n                "Canonical image/label ID mismatch: "\n                f"{image_path.name} vs {label_path.name}"\n            )\n\n        if image_id in images:\n            raise RuntimeError(\n                f"Duplicate canonical patient ID: {image_id}"\n            )\n\n        images[image_id] = image_path\n        labels[image_id] = label_path\n\n        canonical_image_paths.add(image_path.resolve())\n        canonical_label_paths.add(label_path.resolve())\n\n    if len(images) != 484 or len(labels) != 484:\n        raise RuntimeError(\n            "Canonical dataset.json parsing did not produce 484 pairs: "\n            f"images={len(images)}, labels={len(labels)}."\n        )\n\n    disk_images = sorted((DATA_ROOT / "imagesTr").glob("*.nii.gz"))\n    disk_labels = sorted((DATA_ROOT / "labelsTr").glob("*.nii.gz"))\n\n    extra_images = [\n        path\n        for path in disk_images\n        if path.resolve() not in canonical_image_paths\n    ]\n\n    extra_labels = [\n        path\n        for path in disk_labels\n        if path.resolve() not in canonical_label_paths\n    ]\n\n    extra_rows = []\n\n    for path in extra_images:\n        extra_rows.append(\n            {\n                "folder": "imagesTr",\n                "filename": path.name,\n                "status": "not listed in dataset.json training set",\n            }\n        )\n\n    for path in extra_labels:\n        extra_rows.append(\n            {\n                "folder": "labelsTr",\n                "filename": path.name,\n                "status": "not listed in dataset.json training set",\n            }\n        )\n\n    if extra_rows:\n        write_csv(\n            PREP_ROOT / "unlisted_archive_files.csv",\n            extra_rows,\n        )\n\n    print("dataset.json numTraining:", declared)\n    print("Canonical labelled pairs:", len(images))\n    print("Raw imagesTr files on disk:", len(disk_images))\n    print("Raw labelsTr files on disk:", len(disk_labels))\n    print("Unlisted imagesTr files ignored:", len(extra_images))\n    print("Unlisted labelsTr files ignored:", len(extra_labels))\n    print("Canonical image/label pairing: PASS")\n\n    return images, labels\n\n\n# ---------------------------------------------------------------------\n# 5. Deterministic patient-level split\n# ---------------------------------------------------------------------\n\ndef create_split(patient_ids):\n    patient_ids = sorted(patient_ids)\n\n    generator = torch.Generator()\n    generator.manual_seed(SPLIT_SEED)\n\n    permutation = torch.randperm(\n        len(patient_ids),\n        generator=generator,\n    ).tolist()\n\n    shuffled = [patient_ids[i] for i in permutation]\n\n    train_end = TRAIN_CASES\n    val_end = TRAIN_CASES + VAL_CASES\n\n    split = {\n        "train": shuffled[:train_end],\n        "val": shuffled[train_end:val_end],\n        "test": shuffled[val_end:],\n    }\n\n    assert len(split["train"]) == TRAIN_CASES\n    assert len(split["val"]) == VAL_CASES\n    assert len(split["test"]) == TEST_CASES\n\n    assert not (set(split["train"]) & set(split["val"]))\n    assert not (set(split["train"]) & set(split["test"]))\n    assert not (set(split["val"]) & set(split["test"]))\n\n    assert (\n        set(split["train"])\n        | set(split["val"])\n        | set(split["test"])\n    ) == set(patient_ids)\n\n    PREP_ROOT.mkdir(parents=True, exist_ok=True)\n\n    payload = {\n        "split_seed": SPLIT_SEED,\n        "train": split["train"],\n        "val": split["val"],\n        "test": split["test"],\n    }\n\n    (PREP_ROOT / "patient_split.json").write_text(\n        json.dumps(payload, indent=2)\n    )\n\n    print(\n        "Patient split:",\n        {\n            key: len(value)\n            for key, value in split.items()\n        },\n    )\n\n    print("Patient leakage check: PASS")\n\n    return split\n\n\n# ---------------------------------------------------------------------\n# 6. MRI loading and normalization\n# ---------------------------------------------------------------------\n\ndef load_case(image_path, label_path, channel_indices):\n    image_nii = nib.load(str(image_path))\n    label_nii = nib.load(str(label_path))\n\n    image_np = np.asarray(\n        image_nii.dataobj,\n        dtype=np.float32,\n    )\n\n    label_np = np.asarray(\n        label_nii.dataobj,\n        dtype=np.int16,\n    )\n\n    if image_np.ndim != 4:\n        raise RuntimeError(\n            f"Expected 4D image, got {image_np.shape}: {image_path}"\n        )\n\n    modality_axes = [\n        axis\n        for axis, size in enumerate(image_np.shape)\n        if size == 4\n    ]\n\n    if len(modality_axes) != 1:\n        raise RuntimeError(\n            f"Cannot identify modality axis for {image_path}: "\n            f"{image_np.shape}"\n        )\n\n    image_np = np.moveaxis(\n        image_np,\n        modality_axes[0],\n        0,\n    )\n\n    if image_np.shape[1:] != label_np.shape:\n        raise RuntimeError(\n            f"Spatial mismatch for {image_path}: "\n            f"image={image_np.shape}, label={label_np.shape}"\n        )\n\n    image = torch.from_numpy(image_np)\n\n    selected = image[channel_indices].float()\n\n    label = torch.from_numpy(label_np)\n\n    whole_tumour = (label > 0).to(torch.uint8)\n\n    brain = selected.abs().sum(dim=0) > 0\n\n    normalized = torch.zeros_like(selected)\n\n    for channel in range(selected.shape[0]):\n        values = selected[channel][brain]\n\n        if values.numel() == 0:\n            raise RuntimeError(\n                f"Empty non-zero brain region: {image_path}"\n            )\n\n        mu = values.mean()\n        sigma = values.std()\n\n        if sigma <= 0:\n            raise RuntimeError(\n                f"Zero MRI intensity variance: {image_path}"\n            )\n\n        z = (selected[channel] - mu) / sigma\n        z = z.clamp(-5.0, 5.0)\n\n        # Map the clipped z-score range [-5, 5] to [0, 1].\n        normalized[channel][brain] = (\n            z[brain] + 5.0\n        ) / 10.0\n\n    return normalized, whole_tumour, label, brain\n\n\n# ---------------------------------------------------------------------\n# 7. Audit all 484 labels and image geometry\n# ---------------------------------------------------------------------\n\ndef audit_cases(images, labels, split, selected):\n    channel_indices = [\n        selected["FLAIR"],\n        selected["T1-Gd"],\n        selected["T2"],\n    ]\n\n    split_lookup = {}\n\n    for split_name, patients in split.items():\n        for patient_id in patients:\n            split_lookup[patient_id] = split_name\n\n    rows = []\n\n    for index, patient_id in enumerate(sorted(images), start=1):\n        image_nii = nib.load(str(images[patient_id]))\n        label_nii = nib.load(str(labels[patient_id]))\n\n        image_shape = tuple(image_nii.shape)\n        label_shape = tuple(label_nii.shape)\n\n        if len(image_shape) != 4:\n            raise RuntimeError(\n                f"{patient_id}: expected 4D MRI, got {image_shape}"\n            )\n\n        spatial_shape = tuple(\n            size\n            for size in image_shape\n            if size != 4\n        )\n\n        if spatial_shape != label_shape:\n            # Fall back to an explicit modality-axis check.\n            modality_axes = [\n                axis\n                for axis, size in enumerate(image_shape)\n                if size == 4\n            ]\n\n            if len(modality_axes) != 1:\n                raise RuntimeError(\n                    f"{patient_id}: ambiguous MRI modality axis."\n                )\n\n            spatial_shape = tuple(\n                size\n                for axis, size in enumerate(image_shape)\n                if axis != modality_axes[0]\n            )\n\n        if spatial_shape != label_shape:\n            raise RuntimeError(\n                f"{patient_id}: image/label shape mismatch. "\n                f"image={image_shape}, label={label_shape}"\n            )\n\n        label_np = np.asarray(\n            label_nii.dataobj,\n            dtype=np.int16,\n        )\n\n        unique_values = sorted(\n            np.unique(label_np).astype(int).tolist()\n        )\n\n        tumour_voxels = int((label_np > 0).sum())\n        total_voxels = int(label_np.size)\n\n        if tumour_voxels <= 0:\n            raise RuntimeError(\n                f"{patient_id}: no tumour voxels."\n            )\n\n        rows.append(\n            {\n                "patient_id": patient_id,\n                "split": split_lookup[patient_id],\n                "image_shape": "x".join(map(str, image_shape)),\n                "label_shape": "x".join(map(str, label_shape)),\n                "label_values": "|".join(map(str, unique_values)),\n                "tumour_voxels": tumour_voxels,\n                "total_voxels": total_voxels,\n                "tumour_fraction": tumour_voxels / total_voxels,\n            }\n        )\n\n        if index % 50 == 0 or index == len(images):\n            print(f"Audited {index}/{len(images)} patients")\n\n    write_csv(\n        PREP_ROOT / "patient_audit.csv",\n        rows,\n    )\n\n    for split_name in ["train", "val", "test"]:\n        values = [\n            row["tumour_fraction"]\n            for row in rows\n            if row["split"] == split_name\n        ]\n\n        print(\n            f"{split_name:5s} "\n            f"patients={len(values):3d} "\n            f"mean_tumour_fraction={sum(values) / len(values):.6f} "\n            f"min={min(values):.6f} "\n            f"max={max(values):.6f}"\n        )\n\n    print("All-patient label audit: PASS")\n\n    return channel_indices\n\n\n# ---------------------------------------------------------------------\n# 8. Build compact 2D train/validation cache\n# ---------------------------------------------------------------------\n\ndef build_cache_for_patient(\n    split_name,\n    patient_id,\n    image_path,\n    label_path,\n    channel_indices,\n):\n    split_dir = CACHE_ROOT / split_name\n    split_dir.mkdir(parents=True, exist_ok=True)\n\n    output_path = split_dir / f"{patient_id}.pt"\n\n    image, whole_tumour, _, brain = load_case(\n        image_path,\n        label_path,\n        channel_indices,\n    )\n\n    depth = whole_tumour.shape[-1]\n\n    tumour_slices = [\n        z\n        for z in range(depth)\n        if whole_tumour[:, :, z].any().item()\n    ]\n\n    tumour_set = set(tumour_slices)\n\n    brain_slices = [\n        z\n        for z in range(depth)\n        if brain[:, :, z].any().item()\n    ]\n\n    negative_slices = [\n        z\n        for z in brain_slices\n        if z not in tumour_set\n    ]\n\n    positive = evenly_spaced(\n        tumour_slices,\n        POSITIVE_SLICES_PER_PATIENT,\n    )\n\n    negative = evenly_spaced(\n        negative_slices,\n        NEGATIVE_SLICES_PER_PATIENT,\n    )\n\n    if not positive:\n        raise RuntimeError(\n            f"{patient_id}: no positive axial slices."\n        )\n\n    slice_indices = positive + negative\n    positive_flags = (\n        [1] * len(positive)\n        + [0] * len(negative)\n    )\n\n    images_2d = torch.stack(\n        [\n            image[:, :, :, z]\n            for z in slice_indices\n        ],\n        dim=0,\n    ).half()\n\n    masks_2d = torch.stack(\n        [\n            whole_tumour[:, :, z].unsqueeze(0)\n            for z in slice_indices\n        ],\n        dim=0,\n    ).to(torch.uint8)\n\n    if masks_2d.max().item() != 1:\n        raise RuntimeError(\n            f"{patient_id}: cached masks contain no foreground."\n        )\n\n    torch.save(\n        {\n            "patient_id": patient_id,\n            "images": images_2d,\n            "masks": masks_2d,\n            "slice_indices": torch.tensor(\n                slice_indices,\n                dtype=torch.int16,\n            ),\n            "positive": torch.tensor(\n                positive_flags,\n                dtype=torch.uint8,\n            ),\n        },\n        output_path,\n    )\n\n    return {\n        "split": split_name,\n        "patient_id": patient_id,\n        "cached_slices": len(slice_indices),\n        "positive_slices": len(positive),\n        "negative_slices": len(negative),\n        "foreground_fraction": masks_2d.float().mean().item(),\n        "cache_bytes": output_path.stat().st_size,\n    }\n\n\ndef build_train_val_cache(\n    images,\n    labels,\n    split,\n    channel_indices,\n):\n    if CACHE_ROOT.exists():\n        shutil.rmtree(CACHE_ROOT)\n\n    rows = []\n\n    total = TRAIN_CASES + VAL_CASES\n    completed = 0\n\n    for split_name in ["train", "val"]:\n        for patient_id in split[split_name]:\n            row = build_cache_for_patient(\n                split_name=split_name,\n                patient_id=patient_id,\n                image_path=images[patient_id],\n                label_path=labels[patient_id],\n                channel_indices=channel_indices,\n            )\n\n            rows.append(row)\n            completed += 1\n\n            if completed % 25 == 0 or completed == total:\n                print(\n                    f"Cached {completed}/{total} train/val patients"\n                )\n\n    write_csv(\n        PREP_ROOT / "slice_cache_manifest.csv",\n        rows,\n    )\n\n    train_files = list(\n        (CACHE_ROOT / "train").glob("*.pt")\n    )\n\n    val_files = list(\n        (CACHE_ROOT / "val").glob("*.pt")\n    )\n\n    assert len(train_files) == TRAIN_CASES\n    assert len(val_files) == VAL_CASES\n\n    total_bytes = sum(\n        path.stat().st_size\n        for path in train_files + val_files\n    )\n\n    print(\n        "2D cache size:",\n        f"{total_bytes / 1024 ** 3:.2f} GiB",\n    )\n\n    print("Train/validation cache: PASS")\n\n\n# ---------------------------------------------------------------------\n# 9. Preserve original full test volumes\n# ---------------------------------------------------------------------\n\ndef stage_test_volumes(images, labels, split):\n    if TEST_ROOT.exists():\n        shutil.rmtree(TEST_ROOT)\n\n    image_dir = TEST_ROOT / "images"\n    label_dir = TEST_ROOT / "labels"\n\n    image_dir.mkdir(parents=True)\n    label_dir.mkdir(parents=True)\n\n    rows = []\n\n    for index, patient_id in enumerate(\n        split["test"],\n        start=1,\n    ):\n        image_src = images[patient_id]\n        label_src = labels[patient_id]\n\n        image_dst = image_dir / image_src.name\n        label_dst = label_dir / label_src.name\n\n        shutil.copy2(image_src, image_dst)\n        shutil.copy2(label_src, label_dst)\n\n        rows.append(\n            {\n                "patient_id": patient_id,\n                "image": image_dst.name,\n                "label": label_dst.name,\n                "image_sha256": sha256sum(image_dst),\n                "label_sha256": sha256sum(label_dst),\n            }\n        )\n\n        if index % 10 == 0 or index == TEST_CASES:\n            print(f"Staged {index}/{TEST_CASES} test patients")\n\n    write_csv(\n        PREP_ROOT / "test_volume_manifest.csv",\n        rows,\n    )\n\n    assert len(list(image_dir.glob("*.nii.gz"))) == TEST_CASES\n    assert len(list(label_dir.glob("*.nii.gz"))) == TEST_CASES\n\n    print("Full-volume test set staging: PASS")\n\n\n# ---------------------------------------------------------------------\n# 10. Provenance and package\n# ---------------------------------------------------------------------\n\ndef write_metadata(metadata, selected):\n    payload = {\n        "experiment": "EMCAD Experiment 2 - Domain Generalization",\n        "dataset": "Medical Segmentation Decathlon Task01_BrainTumour",\n        "relationship_to_brats": (\n            "Task01_BrainTumour contains the BraTS 2016/2017 cases"\n        ),\n        "download_url": DOWNLOAD_URL,\n        "archive_md5": EXPECTED_MD5,\n        "labelled_cases": 484,\n        "patient_split": {\n            "train": TRAIN_CASES,\n            "val": VAL_CASES,\n            "test": TEST_CASES,\n        },\n        "split_seed": SPLIT_SEED,\n        "target": "Whole Tumour (WT): label > 0",\n        "selected_input_modalities": selected,\n        "cache_policy": {\n            "positive_slices_per_train_val_patient": (\n                POSITIVE_SLICES_PER_PATIENT\n            ),\n            "negative_slices_per_train_val_patient": (\n                NEGATIVE_SLICES_PER_PATIENT\n            ),\n        },\n        "test_policy": (\n            "Full original NIfTI volumes retained for final volumetric testing"\n        ),\n        "dataset_json_name": metadata.get("name"),\n        "dataset_json_modality": metadata.get("modality"),\n        "dataset_json_labels": metadata.get("labels"),\n    }\n\n    (\n        PREP_ROOT / "preparation_metadata.json"\n    ).write_text(\n        json.dumps(payload, indent=2)\n    )\n\n\ndef package_outputs():\n    compact_root = WORK / "EMCAD_BraTS_Part1_Output"\n\n    if compact_root.exists():\n        shutil.rmtree(compact_root)\n\n    shutil.copytree(PREP_ROOT, compact_root)\n\n    archive_path = shutil.make_archive(\n        str(WORK / "EMCAD_BraTS_Part1_Output"),\n        "gztar",\n        root_dir=WORK,\n        base_dir=compact_root.name,\n    )\n\n    checksum_path = WORK / "EMCAD_BraTS_Part1_SHA256SUMS.txt"\n\n    output_files = [\n        Path(archive_path),\n        PREP_ROOT / "patient_split.json",\n        PREP_ROOT / "patient_audit.csv",\n        PREP_ROOT / "slice_cache_manifest.csv",\n        PREP_ROOT / "test_volume_manifest.csv",\n        PREP_ROOT / "preparation_metadata.json",\n    ]\n\n    optional_unlisted = PREP_ROOT / "unlisted_archive_files.csv"\n    if optional_unlisted.is_file():\n        output_files.append(optional_unlisted)\n\n    with checksum_path.open("w") as f:\n        for path in output_files:\n            digest = sha256sum(path)\n            f.write(f"{digest}  {path.name}\\n")\n\n    print()\n    print("=" * 72)\n    print("BRATS PART 1 PREPARATION COMPLETE")\n    print("=" * 72)\n    print("Prepared archive:", archive_path)\n    print(\n        "Prepared archive size:",\n        f"{Path(archive_path).stat().st_size / 1024 ** 3:.2f} GiB",\n    )\n    print("Checksums:", checksum_path)\n    print("No EMCAD training was performed.")\n    print("=" * 72)\n\n\n# ---------------------------------------------------------------------\n# Execute Part 1\n# ---------------------------------------------------------------------\n\ndef main():\n    start = time.time()\n\n    PREP_ROOT.mkdir(parents=True, exist_ok=True)\n\n    download_dataset()\n    extract_labelled_data()\n\n    metadata = read_metadata()\n    selected = resolve_channels(metadata)\n\n    images, labels = collect_cases(metadata)\n\n    split = create_split(images.keys())\n\n    channel_indices = audit_cases(\n        images=images,\n        labels=labels,\n        split=split,\n        selected=selected,\n    )\n\n    build_train_val_cache(\n        images=images,\n        labels=labels,\n        split=split,\n        channel_indices=channel_indices,\n    )\n\n    stage_test_volumes(\n        images=images,\n        labels=labels,\n        split=split,\n    )\n\n    write_metadata(\n        metadata=metadata,\n        selected=selected,\n    )\n\n    package_outputs()\n\n    elapsed = time.time() - start\n\n    print(\n        "Total Part 1 time:",\n        f"{elapsed / 60:.1f} minutes",\n    )\n\n\nif __name__ == "__main__":\n    main()\n'

script_path = Path("/kaggle/working/brats_part1_prepare.py")
script_path.write_text(prep_code)
print("Preparation script:", script_path)
print("Lines:", len(prep_code.splitlines()))


Preparation script: /kaggle/working/brats_part1_prepare.py
Lines: 1079


In [3]:
import py_compile

py_compile.compile(
    "/kaggle/working/brats_part1_prepare.py",
    doraise=True,
)

print("Syntax check: PASS")


Syntax check: PASS


## Run Part 1

This cell performs the complete preparation stage. It is CPU/disk/network work only.

Expected final message:

`BRATS PART 1 PREPARATION COMPLETE`

After it finishes, **Save Version with outputs** in Kaggle.


**If you are rerunning in the same Kaggle session after the 495-file error, the corrected script detects the already-extracted dataset and skips the 7.09 GiB download.**

In [4]:
import subprocess
import sys
from pathlib import Path

stdout_path = Path(
    "/kaggle/working/EMCAD_BraTS_Part1_stdout.log"
)

with stdout_path.open("w") as log:
    process = subprocess.Popen(
        [
            sys.executable,
            "-u",
            "/kaggle/working/brats_part1_prepare.py",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="")
        log.write(line)
        log.flush()

    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"Part 1 failed with return code {return_code}. "
        f"See {stdout_path}"
    )

print("Part 1 execution: PASS")


Free disk before download: 19.50 GiB
Downloading:
https://msd-for-monai.s3-us-west-2.amazonaws.com/Task01_BrainTumour.tar
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0 7255M    0 51718    0     0   131k      0 15:43:42 --:--:-- 15:43:42  131k
  0 7255M    0 16.0M    0     0  11.5M      0  0:10:28  0:00:01  0:10:27 11.5M
  0 7255M    0 57.0M    0     0  23.7M      0  0:05:05  0:00:02  0:05:03 23.7M
  1 7255M    1 87.0M    0     0  25.6M      0  0:04:42  0:00:03  0:04:39 25.6M
  1 7255M    1  117M    0     0  26.7M      0  0:04:31  0:00:04  0:04:27 26.7M
  2 7255M    2  148M    0     0  27.7M      0  0:04:21  0:00:05  0:04:16 29.8M
  2 7255M    2  181M    0     0  28.6M      0  0:04:13  0:00:06  0:04:07 33.3M
  2 7255M    2  215M    0     0  29.2M      0  0:04:07  0:00:07  0:04:00 31.9M
  3 7255

## Verify the saved outputs

Part 2 will use the prepared archive and checksum file from this notebook output. It will not redownload the BraTS-derived dataset.


In [5]:
from pathlib import Path
import json

WORK = Path("/kaggle/working")

required = [
    WORK / "EMCAD_BraTS_Part1_Output.tar.gz",
    WORK / "EMCAD_BraTS_Part1_SHA256SUMS.txt",
    WORK / "EMCAD_BraTS_Part1_stdout.log",
]

for path in required:
    assert path.is_file(), path
    print(
        path.name,
        f"{path.stat().st_size / 1024 ** 2:.2f} MiB",
    )

metadata = json.loads(
    (
        WORK
        / "EMCAD_BraTS_Part1"
        / "preparation_metadata.json"
    ).read_text()
)

print()
print("Dataset:", metadata["dataset"])
print("Target:", metadata["target"])
print("Patient split:", metadata["patient_split"])
print(
    "Input modalities:",
    metadata["selected_input_modalities"],
)

print()
print("PART 1 OUTPUT VERIFICATION: PASS")
print("Save this Kaggle notebook as a Version with outputs.")


EMCAD_BraTS_Part1_Output.tar.gz 628.93 MiB
EMCAD_BraTS_Part1_SHA256SUMS.txt 0.00 MiB
EMCAD_BraTS_Part1_stdout.log 0.03 MiB

Dataset: Medical Segmentation Decathlon Task01_BrainTumour
Target: Whole Tumour (WT): label > 0
Patient split: {'train': 388, 'val': 48, 'test': 48}
Input modalities: {'FLAIR': 0, 'T1-Gd': 2, 'T2': 3}

PART 1 OUTPUT VERIFICATION: PASS
Save this Kaggle notebook as a Version with outputs.
